<a href="https://colab.research.google.com/github/yenlung/Python-Math-AI/blob/main/13%E7%94%A8%E5%85%A8%E9%80%A3%E7%B5%90%E7%A5%9E%E7%B6%93%E7%B6%B2%E8%B7%AF%E5%81%9A%E6%89%8B%E5%AF%AB%E8%BE%A8%E8%AD%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13 用全連結神經網路做手寫辨識 ✍️🧠
## 從零打造神經網路 × MNIST × Gradio Web App

ChatGPT、Gemini、Midjourney 的核心都是**神經網路**。

今天我們要從頭打造一個！

### 這是什麼問題？

給一張 28×28 的手寫數字圖片 → AI 說出「這是 5」

這就是 **MNIST**，AI 的 Hello World！

---

### 神經網路到底在做什麼？

想像你在學認字。老師給你看了幾千個「5」的寫法，你的大腦就「記住」了 5 的特徵。

神經網路做的事情一模一樣：

1. **看數萬張圖片**（訓練數據）
2. **調整內部參數**（學習）
3. **看到新圖片就能判斷**（預測）

---

### 今天做完你會有：

- 🧠 自己訓練的神經網路模型
- 🎨 可以手寫數字讓 AI 辨識的 **Gradio Web App**
- 💡 對深度學習的真實理解

## 🤖 AI 可以怎麼幫你？

你可以問 AI：

- 神經網路的神經元是什麼？
- `relu`、`sigmoid`、`softmax` 有什麼差？
- `batch_size` 和 `epochs` 是什麼意思？
- `loss` 和 `accuracy` 怎麼解讀？
- 怎麼讓模型準確率更高？

但記得：**先自己想，再問 AI**

## 1. 安裝與讀入套件

In [ ]:
!pip install -q gradio

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 神經網路
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD, Adam

# 互動
from ipywidgets import interact, interact_manual

# Web App
import gradio as gr

print(f'TensorFlow 版本: {tf.__version__}')

## 2. 認識 MNIST 數據庫

**MNIST** 是 1998 年誕生的資料集，包含：

- **60,000 張** 訓練圖片
- **10,000 張** 測試圖片
- 每張圖：**28×28 像素**，灰階
- 內容：0 到 9 的手寫數字

當年是用來訓練銀行支票上的數字辨識系統！

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(f'訓練資料: {x_train.shape}  → {len(x_train)} 張，每張 {x_train.shape[1]}×{x_train.shape[2]} 像素')
print(f'測試資料: {x_test.shape}   → {len(x_test)} 張')

### 看看數據長什麼樣

In [ ]:
# 隨機顯示 20 張
fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(20):
    idx = np.random.randint(0, len(x_train))
    axes[i//10][i%10].imshow(x_train[idx], cmap='Greys')
    axes[i//10][i%10].set_title(y_train[idx], fontsize=12)
    axes[i//10][i%10].axis('off')

plt.suptitle('MNIST 隨機樣本（標題為正確答案）', fontsize=13)
plt.tight_layout()
plt.show()

### 圖片的本質：一堆數字！

In [ ]:
n = 0
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 左邊：圖片
axes[0].imshow(x_train[n], cmap='Greys')
axes[0].set_title(f'圖片（正確答案: {y_train[n]}）')
axes[0].axis('off')

# 右邊：數字矩陣（只顯示中間部分）
axes[1].imshow(x_train[n][8:20, 8:20], cmap='Blues')
for i in range(12):
    for j in range(12):
        val = x_train[n][i+8, j+8]
        axes[1].text(j, i, str(val), ha='center', va='center',
                    fontsize=7, color='white' if val > 128 else 'black')
axes[1].set_title('像素值（0=白，255=黑，中間放大）')
axes[1].axis('off')

plt.suptitle('圖片 = 一堆 0~255 的數字', fontsize=13)
plt.tight_layout()
plt.show()

用滑桿隨意看任何一張訓練圖片：

In [ ]:
def show_xy(n=0):
    plt.figure(figsize=(3, 3))
    plt.imshow(x_train[n], cmap='Greys')
    plt.title(f'正確答案: {y_train[n]}')
    plt.axis('off')
    plt.show()

interact_manual(show_xy, n=(0, 59999));

## 3. 整理數據格式

### 輸入：把圖片「攤平」

全連結神經網路只吃「一維向量」，不吃矩陣。

所以每張 28×28 的圖要 reshape 成 784 維的向量：

```
28 × 28 = 784  →  [pixel₁, pixel₂, ..., pixel₇₈₄]
```

另外把像素值從 0~255 縮放到 0~1（讓訓練更穩定）：

In [ ]:
x_train = x_train.reshape(60000, 784) / 255.0
x_test  = x_test.reshape(10000, 784)  / 255.0

print('x_train shape:', x_train.shape)  # (60000, 784)
print('x_test  shape:', x_test.shape)   # (10000, 784)

### 輸出：One-Hot Encoding

神經網路的輸出層有 10 個神經元（對應 0~9），
我們需要把答案轉成「哪個位置是 1」的格式：

```
答案 = 5  →  [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
答案 = 3  →  [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
```

這叫做 **One-Hot Encoding**！

In [ ]:
print('轉換前 y_train[0:5]:', y_train[:5])  # [5 0 4 1 9]

y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

print('轉換後 y_train[0]:', y_train[0])    # [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
print('y_train shape:', y_train.shape)     # (60000, 10)

## 4. 設計神經網路

### 神經元是什麼？

一個神經元做一件事：

$$\text{輸出} = \text{activation}(\mathbf{w} \cdot \mathbf{x} + b)$$

- $\mathbf{x}$：輸入向量
- $\mathbf{w}$：權重（這是 AI **學習**的對象！）
- $b$：偏差值
- activation：激發函數（加入非線性能力）

### 我們的網路架構

```
輸入層      隱藏層 1    隱藏層 2    隱藏層 3    輸出層
784 維  →   20 個    →   20 個    →   20 個    →  10 個
（像素）    (ReLU)      (ReLU)      (ReLU)     (Softmax)
```

輸出的 10 個數字加起來 = 1（機率分佈），代表「這張圖是 0~9 各自的機率」。

In [ ]:
model = Sequential()

# 第 1 個隱藏層：需要說明輸入維度 784
model.add(Dense(20, input_dim=784, activation='relu'))

# 第 2、3 個隱藏層
model.add(Dense(20, activation='relu'))
model.add(Dense(20, activation='relu'))

# 輸出層：10 個分類，用 softmax 變成機率
model.add(Dense(10, activation='softmax'))

### 組裝神經網路

- **loss function**：衡量「預測有多錯」，訓練時要讓它越小越好
- **optimizer**：決定怎麼調整權重（SGD = 隨機梯度下降）
- **metrics**：我們想監控什麼（這裡看正確率）

In [ ]:
model.compile(loss='mse',
              optimizer=SGD(learning_rate=0.087),
              metrics=['accuracy'])

In [ ]:
model.summary()

### 快速算算參數量

| 層 | 連接 | 權重數 | 偏差數 | 小計 |
|----|------|--------|--------|------|
| 輸入 → 隱藏1 | 784 × 20 | 15,680 | 20 | 15,700 |
| 隱藏1 → 隱藏2 | 20 × 20 | 400 | 20 | 420 |
| 隱藏2 → 隱藏3 | 20 × 20 | 400 | 20 | 420 |
| 隱藏3 → 輸出 | 20 × 10 | 200 | 10 | 210 |
| **合計** | | | | **16,750** |

這 16,750 個數字就是 AI 要「學習」的！

## 5. 訓練！

- `epochs`：整個訓練資料看幾遍（越多越可能學得好，但也可能過擬合）
- `batch_size`：一次餵幾筆數據（預設 32）
- `validation_split`：保留 10% 訓練資料當驗證集（監控學習狀況）

⏱️ 有點耐心，訓練需要幾十秒！

In [ ]:
history = model.fit(x_train, y_train,
                    epochs=20,
                    batch_size=128,
                    validation_split=0.1,
                    verbose=1)

### 觀察學習過程

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 正確率
axes[0].plot(history.history['accuracy'], label='訓練集', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='驗證集', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('訓練過程 — 正確率')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='訓練集', linewidth=2)
axes[1].plot(history.history['val_loss'], label='驗證集', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('訓練過程 — Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. 評估結果

In [ ]:
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f'測試集 Loss:    {loss:.4f}')
print(f'測試集正確率: {acc*100:.2f}%')
print(f'10,000 張圖中答對了 {int(acc*10000)} 張！')

In [ ]:
# 對測試集全部預測
predict = np.argmax(model.predict(x_test, verbose=0), axis=-1)
y_true  = np.argmax(y_test, axis=-1)

### 互動：用滑桿查看任意測試圖片的預測

In [ ]:
def show_prediction(測試編號=0):
    img = x_test[測試編號].reshape(28, 28)
    pred = predict[測試編號]
    true = y_true[測試編號]

    plt.figure(figsize=(3, 3))
    plt.imshow(img, cmap='Greys')
    color = 'green' if pred == true else 'red'
    plt.title(f'AI 判斷: {pred}  |  正確答案: {true}',
              color=color, fontsize=13)
    plt.axis('off')
    plt.show()

interact_manual(show_prediction, 測試編號=(0, 9999));

### 找出 AI 猜錯的圖片

In [ ]:
wrong_indices = np.where(predict != y_true)[0]
print(f'共 {len(wrong_indices)} 張圖片猜錯了（正確率 {(1 - len(wrong_indices)/10000)*100:.2f}%）')

# 顯示前 20 個錯誤
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))

for i, idx in enumerate(wrong_indices[:20]):
    axes[i//10][i%10].imshow(x_test[idx].reshape(28, 28), cmap='Greys')
    axes[i//10][i%10].set_title(f'預:{predict[idx]}\n真:{y_true[idx]}',
                                fontsize=9, color='red')
    axes[i//10][i%10].axis('off')

plt.suptitle('AI 猜錯的圖片（預測/真實）', fontsize=13)
plt.tight_layout()
plt.show()

## 7. 打造 Gradio 手寫辨識 Web App！🎉

這是最酷的部分——讓任何人都能用瀏覽器來測試你的 AI！

In [ ]:
def resize_image(inp):
    """把 Gradio 畫板的圖轉成 MNIST 格式的 784 維向量"""
    # inp["layers"][0] 是畫板的第一層（RGBA 格式）
    image = np.array(inp["layers"][0], dtype=np.float32).astype(np.uint8)
    image_pil = Image.fromarray(image)

    # 轉成白底 RGB
    background = Image.new("RGB", image_pil.size, (255, 255, 255))
    background.paste(image_pil, mask=image_pil.split()[3])
    image_pil = background

    # 灰階 → 縮小到 28×28
    image_gray = image_pil.convert("L")
    img_array = np.array(image_gray.resize((28, 28), resample=Image.LANCZOS))

    # 反轉（MNIST 是黑字白底→白字黑底），攤平，縮放
    img_array = (255 - img_array).reshape(1, 784) / 255.0
    return img_array


def recognize_digit(inp):
    """輸入畫板圖片，輸出各數字的機率"""
    img_array = resize_image(inp)
    prediction = model.predict(img_array, verbose=0).flatten()
    labels = list('0123456789')
    return {labels[i]: float(prediction[i]) for i in range(10)}

In [ ]:
iface = gr.Interface(
    fn=recognize_digit,
    inputs=gr.Sketchpad(label="在這裡手寫一個數字"),
    outputs=gr.Label(num_top_classes=3, label="AI 判斷結果"),
    title="✍️ MNIST 手寫數字辨識",
    description="在畫板上寫一個 0~9 的數字，AI 會告訴你它認為是什麼！",
    theme="soft"
)

iface.launch(share=True, debug=True)

## 💡 改進挑戰

現在的模型正確率大約在 **90~93%**。能不能讓它更高？

試試以下幾個方向（問 AI 怎麼做）：

| 方向 | 做法 | 預期效果 |
|------|------|----------|
| 加更多神經元 | `Dense(128, ...)` | ⬆️ 可能更準，但更慢 |
| 換 optimizer | `optimizer='adam'` | ⬆️ 通常比 SGD 更快收斂 |
| 加更多 epochs | `epochs=50` | ⬆️ 有機會更準，小心過擬合 |
| 換 loss | `categorical_crossentropy` | ⬆️ 分類問題更適合 |
| 加 Dropout 層 | `Dropout(0.2)` | ⬆️ 防止過擬合 |

In [ ]:
# TODO：試試更強的模型
# 提示：把 layer size 改大，換成 adam optimizer 試試

from tensorflow.keras.layers import Dropout

model2 = Sequential([
    Dense(128, input_dim=784, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

model2.compile(loss='categorical_crossentropy',
               optimizer='adam',
               metrics=['accuracy'])

history2 = model2.fit(x_train, y_train,
                      epochs=15,
                      batch_size=128,
                      validation_split=0.1,
                      verbose=1)

_, acc2 = model2.evaluate(x_test, y_test, verbose=0)
print(f'改進版正確率: {acc2*100:.2f}%')
print(f'原始版正確率: {acc*100:.2f}%')

# 🧠 核心觀念回顧

```
全連結神經網路的流程

圖片 (28×28)
  ↓ reshape
向量 (784)
  ↓ 全連結層 × 3（ReLU）
隱藏特徵
  ↓ 輸出層（Softmax）
機率分佈 (10)  →  取最大值 → 預測答案
```

| 步驟 | 關鍵操作 | 說明 |
|------|----------|------|
| 數據前處理 | reshape + /255 | 攤平 + 縮放到 0~1 |
| 標籤處理 | to_categorical | One-Hot Encoding |
| 建模 | Sequential + Dense | 堆疊全連結層 |
| 組裝 | compile | 指定 loss + optimizer |
| 訓練 | fit | 看數據、調參數 |
| 評估 | evaluate | 在新數據上測試 |

# 🎯 本週創作任務

### 任務 A：調出更好的模型
用自己的想法調整神經網路架構，讓測試集正確率超過 **97%**！
（記得用 `validation_split` 監控，不要只看訓練集！）

### 任務 B：客製化你的 Gradio App
改造 Gradio 介面：
- 換一個更有趣的標題和說明
- 加入「清除」按鈕
- 加入「給個範例看看」的例子圖片

### 任務 C（進階）：手寫字母辨識
EMNIST 資料集包含手寫字母，試試訓練一個字母辨識器！
（問 AI：怎麼用 `emnist` 資料集？）

回答：

1️⃣ 你最後的模型架構是什麼？正確率多少？
2️⃣ 你改了什麼讓模型變好？
3️⃣ AI 如何幫助你完成？